# TERT promoter — chr5:1295046 discovery

Reproduces the MCP `discover_variant` call. Scores every track
on the **alphagenome** oracle for the variant
**chr5:1295046 T>G** and returns the
top tracks per regulatory layer.


In [ ]:
# All imports the notebook needs — top-level, no later imports.
import json
from pathlib import Path

import chorus
from chorus.analysis.normalization import get_normalizer
from chorus.analysis.discovery import discover_variant_effects

In [ ]:
WALKTHROUGH_DIR = Path.cwd()
print(f"Writing artifacts to: {WALKTHROUGH_DIR}")

In [ ]:
# Instantiate + load the oracle in its subprocess env.
oracle = chorus.create_oracle(
    oracle_name='alphagenome',
    use_environment=True,
)
oracle.load_pretrained_model()

In [ ]:
# Inputs.
oracle_name = 'alphagenome'
variant_position = 'chr5:1295046'
alleles = ['T', 'G']
gene_name = 'TERT'
top_n_per_layer = 3
top_n_cell_types = 10
ranking_metric = "alt_x_abs_effect"
min_ref_value = 0.0

In [ ]:
# Discovery scoring across ALL tracks (no assay_ids whitelist).
# Saves the IGV report directly into the walkthrough dir.
normalizer = get_normalizer(oracle_name=oracle_name)
discovery_result = discover_variant_effects(
    oracle=oracle,
    oracle_name=oracle_name,
    variant_position=variant_position,
    alleles=alleles,
    top_n_per_layer=top_n_per_layer,
    top_n_cell_types=top_n_cell_types,
    gene_name=gene_name,
    normalizer=normalizer,
    output_path=str(WALKTHROUGH_DIR),
    output_filename='chr5_1295046_T_G_TERT_alphagenome_report.html',
    igv_raw=False,
    analysis_request=None,
    ranking_metric=ranking_metric,
    min_ref_value=min_ref_value,
)

In [ ]:
# Save markdown summary + structured JSON. discover_variant_effects already
# wrote the HTML in the previous cell.
report = discovery_result.pop("report", None)
if report is not None:
    WALKTHROUGH_DIR.joinpath("example_output.md").write_text(report.to_markdown())
WALKTHROUGH_DIR.joinpath("example_output.json").write_text(
    json.dumps(discovery_result, indent=2, default=str)
)
try:
    if report is not None:
        report.to_dataframe().to_csv(
            WALKTHROUGH_DIR / "example_output.tsv", sep="\t", index=False,
        )
except Exception as exc:
    print(f"TSV write skipped: {exc}")
print(f"Wrote artifacts to {WALKTHROUGH_DIR}")

## What this notebook produced

- `example_output.md`, `.json`, `.tsv` — top tracks per layer + cell-type ranking
- `chr5_1295046_T_G_TERT_alphagenome_report.html` — interactive IGV report for the selected top tracks
